## Computational Aspects of Complex Reflection Groups

Götz Pfeiffer - University of Galway

# 5. More Computing: Advanced Core Topics in CGT

<div class="alert alert-danger">

**Draft.**  This notebook is a draft of Part 5, not yet part of the collection.  It covers four core topics of Computational Group Theory that Parts 1–4 leave out, each as a continuation of one of them.

</div>

* **Schreier-Sims** (continues *Stabilizer Chain*, Part 1): a stabilizer chain with a small set of generators, and **sifting**.
* **Backtrack search** (continues *Tree Traversal*, Part 1): depth-first search through the elements of a group, for centralizers and intersections.
* **Presentations of subgroups** (continues *Coset Enumeration*, Part 3): the **Reidemeister-Schreier** method.
* **The MeatAxe** (continues *Spinning*, Part 4): testing a module for irreducibility.

The references, like [HEO05], are those of the preface.

In [ ]:
using OrbitAl: Perm, orbit, transpositions, isidentity, last_moved, onPoints, cycles
using OrbitAl: presentations
using OrbitAl.enumerator: coset_table, perms
using LinearAlgebra
import OrbitAl

## Schreier-Sims

* Recall from Part 1 how `sizeOfGroup` works its way along a chain of stabilizers $G \geq G_x \geq G_{x,y} \geq \dots$, with Schreier generators for each stabilizer.
* The number of Schreier generators grows quickly from one link to the next: with $|A|$ generators and an orbit of length $n$, there are up to $n |A|$ of them, most of them redundant.
* The Schreier-Sims algorithm [Sim70] keeps only the Schreier generators that are really needed.  It is the basis of most algorithms for permutation groups [Ser03], [HEO05].

<div class="alert alert-danger">

**Definition.**  Let $G$ be a group of permutations of $\{1, \dots, n\}$.
* A **base** of $G$ is a list $B = (b_1, \dots, b_k)$ of points such that only the identity of $G$ fixes all of them.
* It defines a **stabilizer chain** $G = G^{(1)} \geq G^{(2)} \geq \dots \geq G^{(k+1)} = 1$, where $G^{(i+1)}$ is the stabilizer of $b_i$ in $G^{(i)}$.
* A **strong generating set** of $G$ (relative to $B$) is a generating set $S$ of $G$ such that $S \cap G^{(i)}$ generates $G^{(i)}$ for each $i$.

</div>

* For each link $i$ of the chain, we store the base point $b_i$, the generators $S^{(i)} = S \cap G^{(i)}$, the orbit of $b_i$ under $G^{(i)}$, and a transversal: for each point $y$ in the orbit an element $t_y \in G^{(i)}$ with $b_i^{t_y} = y$.
* The orbit and the transversal are computed by the orbit algorithm, with a dictionary in place of the list `reps` of `orbit_with_transversal`.

In [ ]:
mutable struct Link
    point::Int
    gens::Vector{Perm}
    orbit::Vector{Int}
    reps::Dict{Int, Perm}
end

function extend!(link::Link)
    id = link.reps[link.point]
    link.orbit = [link.point]
    link.reps = Dict(link.point => id)
    for y in link.orbit, a in link.gens
        z = y^a
        haskey(link.reps, z) || begin
            push!(link.orbit, z)
            link.reps[z] = link.reps[y] * a
        end
    end
    return link
end

Link(point, gens, id) = extend!(Link(point, gens, [point], Dict(point => id)))

### Sifting

* Each $g \in G$ is uniquely a product $g = u_k \cdots u_2 u_1$ with $u_i$ in the transversal of link $i$: if $b_1^g = y$ then $g t_y^{-1} \in G^{(2)}$, and so on.
* **Sifting** an arbitrary permutation $g$ follows this decomposition as far as it goes.  It stops at the first link $i$ where $b_i^g$ is not in the orbit, and returns the remaining permutation, the **residue**, together with $i$.
* So $g \in G$ if and only if $g$ sifts all the way through, with residue $1$.

In [ ]:
function sift(chain, g, i = 1)
    while i <= length(chain)
        link = chain[i]
        y = link.point^g
        haskey(link.reps, y) || return g, i
        g = g / link.reps[y]
        i += 1
    end
    return g, i
end

### The Algorithm

<div class="alert alert-info">

**Schreier-Sims Algorithm**
* **Input:** a list $A$ of generators of a permutation group $G$.
* **Output:** a base $B$ and a strong generating set $S$ of $G$, as a stabilizer chain.
* **1.** Start with a chain of a single link, with all generators, and a base point moved by one of them.
* **2.** At link $i$, sift each Schreier generator $t_y a t_{y.a}^{-1}$ through the links after $i$.
  * If all of them sift through, link $i$ is complete: go back to link $i-1$.
  * If one of them leaves a residue $h \neq 1$ at link $j$, add $h$ to the generators of the links $i+1, \dots, j$ (adding a new base point moved by $h$, if $j$ is new), and continue at link $j$.
* **3.** Stop when link $1$ is complete.

</div>

* By Schreier's Lemma, when link $i$ is complete, the generators of the links $i+1, \dots$ generate the stabilizer of $b_i$ in the group generated by those of link $i$ [HEO05], [Ser03].
* `schreier_sims` accepts an initial part of the base, which the backtrack search below will use.

In [ ]:
function schreier_test(chain, i)
    link = chain[i]
    for y in link.orbit, a in link.gens
        s = link.reps[y] * a / link.reps[y^a]
        h, j = sift(chain, s, i + 1)
        isidentity(h) || return h, j
    end
    return nothing, 0
end

function schreier_sims(gens, id; base = Int[])
    gens = filter(!isidentity, gens)
    isempty(base) && !isempty(gens) && (base = [last_moved(gens[1])])
    chain = [Link(b, l == 1 ? copy(gens) : Perm[], id) for (l, b) in enumerate(base)]
    i = 1
    while 1 <= i <= length(chain)
        h, j = schreier_test(chain, i)
        if isnothing(h)
            i -= 1
        else
            j > length(chain) && push!(chain, Link(last_moved(h), Perm[], id))
            for l in i+1:j
                push!(chain[l].gens, h)
                extend!(chain[l])
            end
            i = j
        end
    end
    return chain
end

<div class="alert alert-warning">

<img src="images/julia.png" alt="julia" width="30">

* `1 <= i <= length(chain)` is a **chained comparison**: it means `1 <= i && i <= length(chain)`.
* `isnothing(h)` tests whether `schreier_test` found nothing.

</div>

### Order, Membership, Random Elements

* The functions `sizeOfGroup`, `memberOfGroup` and `randomGroupElement` of Part 1 become much simpler with a stabilizer chain.
* By the decomposition $g = u_k \cdots u_1$, the order of $G$ is the product of the orbit lengths.
* Membership is sifting.  And a product of random transversal elements is a random element of $G$, uniformly distributed.

In [ ]:
sizeOfGroup(chain::Vector{Link}) = prod(big(length(link.orbit)) for link in chain; init = big(1))
memberOfGroup(chain::Vector{Link}, g) = isidentity(first(sift(chain, g)))
randomGroupElement(chain::Vector{Link}, id) = prod(rand(collect(values(link.reps))) for link in reverse(chain); init = id)
base(chain) = [link.point for link in chain]
strong_gens(chain) = unique(vcat([link.gens for link in chain]...))

<div class="alert alert-warning">

<img src="images/julia.png" alt="julia" width="30">

* `big(n)` is `n` as a `BigInt`, an integer of arbitrary size: the order of the Rubik's cube group does not fit into an `Int`.

</div>

* OrbitAl wraps a stabilizer chain into a group type, `SimsGp`: a list of generators, together with a chain that is computed when it is first needed.  Then `size`, `in` and `rand` work on the chain.

In [ ]:
mutable struct SimsGp
    gens::Vector{Perm}
    one::Perm
    chain::Union{Nothing, Vector{Link}}
    SimsGp(gens, one) = new(gens, one, nothing)
end

function stabChain(G::SimsGp)
    isnothing(G.chain) && (G.chain = schreier_sims(G.gens, G.one))
    return G.chain
end

Base.size(G::SimsGp) = sizeOfGroup(stabChain(G))
Base.in(g::Perm, G::SimsGp) = memberOfGroup(stabChain(G), g)
Base.rand(G::SimsGp) = randomGroupElement(stabChain(G), G.one)

### Examples

* The Mathieu group $M_{24}$, as in Part 1, and the Rubik's cube group [Joy08], from the example groups of OrbitAl.

In [ ]:
using OrbitAl.permgroups: m24, cube

In [ ]:
for (name, gens, id) in [("S10", transpositions(10), Perm(10)), ("M24", m24, Perm(24)), ("cube", cube, Perm(48))]
    chain = schreier_sims(gens, id)
    println(name, ": order ", sizeOfGroup(chain), ", base of length ", length(chain),
            ", ", length(strong_gens(chain)), " strong generators")
end

In [ ]:
chain = schreier_sims(m24, Perm(24))
all(memberOfGroup(chain, randomGroupElement(chain, Perm(24))) for _ in 1:100), count(memberOfGroup(chain, rand(Perm, 24)) for _ in 1:100)

* The Rubik's cube group, as a `SimsGp`:

In [ ]:
rubik = SimsGp(cube, Perm(48))
size(rubik), rand(rubik) in rubik, Perm(48, [[1, 3]]) in rubik

## Backtrack Search

* Through the cosets of its stabilizers, a stabilizer chain turns the elements of $G$ into a **tree** of depth $k$.  The nodes at depth $i$ are the right cosets $G^{(i+1)} g$, for $g \in G$, and the children of a node are the cosets of $G^{(i+2)}$ that it contains.  The root is $G$ itself, and the leaves are the elements of $G$.
* By the decomposition $g = u_k \cdots u_1$, the node $G^{(i+1)} g$ is represented by the partial product $u_i \cdots u_1$, and its children by the products $u_{i+1} u_i \cdots u_1$, for the elements $u_{i+1}$ of the transversal of link $i+1$.
* All elements below a node have the same images $b_1^g, \dots, b_i^g$ of the first $i$ base points: the later factors fix them.
* **Backtrack search** traverses this tree depth first (DFS, Part 1), to find the elements of $G$ with some property $P$.
* Its power lies in **pruning**: if the images of $b_1, \dots, b_i$ already rule out $P$, the whole subtree below the node can be skipped [But91], [Ser03], [HEO05].

In [ ]:
function backtrack(chain, id, accept, prune = (g, i) -> true)
    found = Perm[]
    nodes = 0
    function search(g, i)       # g fixes the images of b_1, ..., b_{i-1}
        nodes += 1
        if i > length(chain)
            accept(g) && push!(found, g)
            return
        end
        link = chain[i]
        for y in link.orbit
            h = link.reps[y] * g
            prune(h, i) && search(h, i + 1)
        end
    end
    search(id, 1)
    return (found = found, nodes = nodes)
end

<div class="alert alert-warning">

<img src="images/julia.png" alt="julia" width="30">

* The local function `search` can change the variable `nodes` of the enclosing function, and call itself recursively.

</div>

### Centralizers

* $g$ centralizes $x$ if and only if $(p^x)^g = (p^g)^x$ for all points $p$.
* If both $b_j$ and $b_j^x$ are among the first $i$ base points, this condition can be checked at depth $i$ already.
* The test prunes best when the base follows the cycles of $x$.

In [ ]:
function centralizer(gens, id, x)
    supp = vcat([c for c in cycles(x) if length(c) > 1]...)
    chain = schreier_sims(gens, id; base = supp)
    B = base(chain); pos = Dict(b => l for (l, b) in enumerate(B))
    prune(h, i) = all((B[j]^h)^x == B[pos[B[j]^x]]^h for j in 1:i if get(pos, B[j]^x, i + 1) <= i)
    backtrack(chain, id, g -> x^g == x, prune)
end

In [ ]:
S6 = transpositions(6); x = Perm(6, [[1, 2], [3, 4]])
c = centralizer(S6, Perm(6), x)
length(c.found), c.nodes, backtrack(schreier_sims(S6, Perm(6)), Perm(6), g -> x^g == x).nodes

* The centralizer of an element of order 23 in $M_{24}$, found among $|M_{24}| = 244823040$ elements:

In [ ]:
c = centralizer(m24, Perm(24), m24[1])
length(c.found), c.nodes

### Intersections

* To find $G \cap H$, search the tree of $G$, and test the leaves for membership in $H$.
* Give $H$ a stabilizer chain with the same base as $G$.  Then a node at depth $i$ can only lead to elements of $H$ if it sifts through the first $i$ links of $H$'s chain.
* For example, $A_4 \cap D_8$, as in Exercise 1.26:

In [ ]:
function intersection(Ggens, Hgens, id)
    G = schreier_sims(Ggens, id)
    H = schreier_sims(Hgens, id; base = base(G))
    prune(h, i) = sift(H[1:i], h)[2] > i
    backtrack(G, id, g -> memberOfGroup(H, g), prune)
end

In [ ]:
a, b = Perm([2,3,1,4]), Perm([2,1,4,3])     # A4
r, s = Perm([2,3,4,1]), Perm([3,2,1,4])     # D8
sort(intersection([a, b], [r, s], Perm(4)).found)

* This search lists **all** elements of $G \cap H$.  It is better to search only for **generators**, and to let the subgroup found so far skip whole subtrees.

### Generators of a Subgroup

* Let $K = \{g \in G : P(g)\}$ be a subgroup of $G$, such as $G \cap H$ or a centralizer, and let $K^{(i)} = K \cap G^{(i)}$.  We find generators of $K^{(i)}$ for $i = k, \dots, 2, 1$, from the end of the chain back to its start.
* $K^{(i+1)}$ is the stabilizer of $b_i$ in $K^{(i)}$.  So $K^{(i)}$ is generated by the generators of $K^{(i+1)}$, together with elements of $K^{(i)}$ whose images of $b_i$ make up the orbit $b_i^{K^{(i)}}$.
* The elements of $G^{(i)}$ that map $b_i$ to $y$ form the coset $G^{(i+1)} t_y$, a node at depth $i$ of the tree, represented by $t_y$.  `first_leaf` searches the leaves below it depth first, and stops at the first leaf with property $P$.
* If $y$ is already in the orbit of $b_i$ under the generators found so far, then the whole subtree below this node can be skipped.

In [ ]:
function first_leaf(chain, g, i, accept, prune)    # g fixes the images of b_1, ..., b_{i-1}
    i > length(chain) && return accept(g) ? g : nothing
    link = chain[i]
    for y in link.orbit
        h = link.reps[y] * g
        prune(h, i) || continue
        f = first_leaf(chain, h, i + 1, accept, prune)
        isnothing(f) || return f
    end
    return nothing
end

function subgp_gens(chain, accept, prune = (g, i) -> true)
    gens = Perm[]
    for i in length(chain):-1:1
        link = chain[i]
        for y in link.orbit
            y in orbit(gens, link.point, onPoints) && continue
            t = link.reps[y]
            prune(t, i) || continue
            g = first_leaf(chain, t, i + 1, accept, prune)
            isnothing(g) || push!(gens, g)
        end
    end
    return gens
end

<div class="alert alert-warning">

<img src="images/julia.png" alt="julia" width="30">

* `for i in length(chain):-1:1` counts down, in steps of $-1$.

</div>

* For $G \cap H$, the property is membership in $H$, and the pruning is that of `intersection`.

In [ ]:
function Base.intersect(G::SimsGp, H::SimsGp)
    chG = stabChain(G)
    chH = schreier_sims(H.gens, H.one; base = base(chG))
    prune(h, i) = sift(chH[1:i], h)[2] > i
    return SimsGp(subgp_gens(chG, g -> memberOfGroup(chH, g), prune), G.one)
end

In [ ]:
K = SimsGp([a, b], Perm(4)) ∩ SimsGp([r, s], Perm(4))
K.gens, size(K)

* The stabilizer of the point $24$ in $M_{24}$ is the Mathieu group $M_{23}$, the intersection of $M_{24}$ with the symmetric group on $23$ points:

In [ ]:
S23 = SimsGp([Perm(24, [[i, i + 1]]) for i in 1:22], Perm(24))
M23 = SimsGp(m24, Perm(24)) ∩ S23
length(M23.gens), size(M23)

* Practical implementations also refine the images of the base points with ordered partitions [Leo91], [HEO05], [Ser03].

## Presentations of Subgroups

* Let $G = \langle S \mid R \rangle$, with $S = S^{-1}$, and let $H \leq G$ have finite index.  A coset enumeration (Part 3) gives the action of $G$ on the cosets of $H$.
* $H$ is the stabilizer of the coset $H$ in this action.  So by Schreier's Lemma (Part 1) it is generated by the **Schreier generators** $t_x s t_{x.s}^{-1}$, where the $t_x$ form a transversal.
* Take as transversal the words found by BFS from the coset $H$: they form a **Schreier transversal**, closed under taking prefixes.  The Schreier generators of the edges of the BFS tree are trivial.
* The others are generators of $H$, as an abstract group, and rewriting the relators gives relators for them.

<div class="alert alert-danger">

**Theorem** (Reidemeister-Schreier)**.**  With a Schreier transversal $\{t_x\}$, the group $H$ has a presentation with generators $\gamma(x, s) = t_x s t_{x.s}^{-1}$ for the non-tree edges $x \stackrel{s}{\longrightarrow} x.s$, and relators: the words $t_x r t_x^{-1}$, for each coset $x$ and each relator $r \in R$, rewritten in the $\gamma(x, s)$ [Rei26], [Sch27], [Sim94], [HEO05].

</div>

* For $S = S^{-1}$, the edges come in pairs $x \stackrel{s}{\longrightarrow} y$ and $y \stackrel{s^{-1}}{\longrightarrow} x$, with $\gamma(y, s^{-1}) = \gamma(x, s)^{-1}$.  So each pair gives one generator $k$, written $k$ and $-k$.  If $s$ is an involution that fixes $x$, then $\gamma(x, s)$ is an involution itself.

In [ ]:
coset_perms(genrel, sbgp) = perms(coset_table(genrel, sbgp))

function reidemeister_schreier(genrel, sbgp)
    P = coset_perms(genrel, sbgp)
    invr = genrel.invr
    # Schreier transversal: BFS words, and the tree edges
    word = Dict(1 => Int[]); list = [1]; tree = Set{Tuple{Int,Int}}()
    for x in list, s in genrel.gens
        y = x^P[s]
        haskey(word, y) && continue
        word[y] = [word[x]; s]; push!(list, y)
        push!(tree, (x, s), (y, invr[s]))
    end
    # Schreier generators: one for each pair of non-tree edges
    label = Dict{Tuple{Int,Int},Int}(); edges = Tuple{Int,Int}[]; rels = Vector{Vector{Int}}()
    for x in list, s in genrel.gens
        ((x, s) in tree || haskey(label, (x, s))) && continue
        push!(edges, (x, s)); k = length(edges)
        label[(x, s)] = k
        partner = (x^P[s], invr[s])
        if partner == (x, s)
            push!(rels, [k, k])       # an involution
        else
            label[partner] = -k
        end
    end
    # rewrite the relators t_x r t_x^-1
    relators = [[l; reverse(invr[r])] for (l, r) in genrel.rels]
    for x in list, w in relators
        rel = Int[]; y = x
        for s in w
            haskey(label, (y, s)) && push!(rel, label[(y, s)])
            y = y^P[s]
        end
        isempty(rel) || push!(rels, rel)
    end
    return (index = length(list), word = word, edges = edges, rels = rels, perms = P)
end

* For example, the symmetric group $S_4$ (type $A_3$) and its subgroup $S_3$:

In [ ]:
G = presentations.A3
rs = reidemeister_schreier(G, G.sbgp)
rs.index, length(rs.edges), length(rs.rels)

* Check 1: evaluate the new generators in $G$ (from a coset enumeration over the trivial subgroup), and the relators on them.
* Check 2: enumerate the cosets of the trivial subgroup in the new presentation.  Its order should be $|G|/|G:H|$.

In [ ]:
function check(genrel, rs)
    Q = coset_perms(genrel, Vector{Int}[])            # regular representation of G
    t(x) = prod(Q[rs.word[x]]; init = one(Q[1]))
    val = [t(x) * Q[s] / t(x^rs.perms[s]) for (x, s) in rs.edges]
    ev(w) = prod((k > 0 ? val[k] : inv(val[-k]) for k in w); init = one(Q[1]))
    # the presentation of H as input for the coset enumerator
    m = length(rs.edges)
    selfinv = Set(r[1] for r in rs.rels if length(r) == 2 && r[1] == r[2])
    letter(k) = k > 0 || -k in selfinv ? abs(k) : m - k
    invr = [k in selfinv ? k : (k <= m ? m + k : k - m) for k in 1:2m]
    pres = (gens = collect(1:2m), invr = invr, rels = [(letter.(r), Int[]) for r in rs.rels])
    order_H = coset_table(pres, Vector{Int}[]).active
    (relators_hold = all(isidentity(ev(w)) for w in rs.rels), order_G = length(Q[1].list), order_H = order_H)
end

In [ ]:
check(G, rs)

* The subgroup $\langle t_0, t_1, t_2 \rangle$ of $G(3,3,3)$ from Part 3:

In [ ]:
G = presentations.G333
rs = reidemeister_schreier(G, G.sbgp)
(rs.index, length(rs.edges), length(rs.rels)), check(G, rs)

* These presentations are very redundant.  **Tietze transformations** remove superfluous generators and relators [Sim94], [HEO05].

## The MeatAxe

* Spinning (Part 4) finds the submodule generated by a single vector.  The **MeatAxe** [Par84] uses this to decide whether a module is irreducible, and to split it if not.
* It works over finite fields.  The field $\mathbb{F}_7$ is the type `Zn{7}` from OrbitAl: the residues modulo $7$.
* Then the linear Union-Find and spinning of Part 4 work over $\mathbb{F}_7$ unchanged: they only need `+`, `*`, `zero`, `one` and `inv` of the coefficients.  Here, they are taken from OrbitAl.

In [ ]:
using OrbitAl.modn: Zn
const F = Zn{7}

using OrbitAl: onRight
using OrbitAl.sparsevec, OrbitAl.unionfind
using OrbitAl.linear: spinning

<div class="alert alert-warning">

<img src="images/julia.png" alt="julia" width="30">

* In `Zn{n}`, the modulus `n` is a **type parameter**: `Zn{7}` and `Zn{5}` are different types, and a matrix over `Zn{7}` knows its modulus.  The constructor `Zn{n}(x)` reduces `x` modulo `n`.
* Integers mix with residues: `F(3) + 5 == F(1)`.
* `const` declares a global constant.

</div>

* To find a candidate vector, the MeatAxe takes a random element $\theta$ of the algebra generated by the matrices, and spins the vectors in its null space.

<div class="alert alert-danger">

**Theorem** (Norton's irreducibility test)**.**  Let $V$ be a module for an algebra $A$, and let $\theta \in A$ have a null space of dimension $1$, spanned by $v$, and let $w$ span the null space of $\theta^T$.  Then $V$ is irreducible if and only if $v$ spins to $V$ under $A$, and $w$ spins to the dual space under the transposed matrices.

</div>

* *Why:* if $U$ is a proper submodule, then $\theta$ is singular on $U$ or on $V/U$.  In the first case $v \in U$.  In the second case $w$ lies in the annihilator of $U$, which is a proper submodule of the dual module.
* $\theta$ is a random combination of a few random products of the generators, shifted by a scalar $\lambda$ so as to become singular.  Holt and Rees [HR94] extend the test to null spaces of larger dimension.

In [ ]:
function left_nullspace(A)          # a basis of {v : v A = 0}, as row vectors
    n, m = size(A)
    T = eltype(A)
    rows = [[A[i, :]; [j == i ? one(T) : zero(T) for j in 1:n]] for i in 1:n]    # [A | I]
    r = 0
    for col in 1:m
        i = findfirst(k -> k > r && !iszero(rows[k][col]), 1:n)
        isnothing(i) && continue
        r += 1; rows[r], rows[i] = rows[i], rows[r]
        rows[r] = inv(rows[r][col]) .* rows[r]
        for k in 1:n
            k != r && !iszero(rows[k][col]) && (rows[k] = rows[k] - rows[k][col] .* rows[r])
        end
    end
    [permutedims(row[m+1:end]) for row in rows[r+1:end]]
end

function norton(gens; tries = 50)
    n = size(gens[1], 1)
    for _ in 1:tries
        words = [prod(rand(gens, rand(1:4))) for _ in 1:3]
        θ0 = sum(rand(F.(0:6)) * w for w in words)
        for λ in F.(0:6)
            θ = θ0 - λ * I
            N = left_nullspace(θ)
            length(N) == 1 || continue
            U = spinning(gens, N[1], onRight)
            length(U) < n && return (irreducible = false, submodule = U)
            W = spinning(transpose.(gens), left_nullspace(transpose(θ))[1], onRight)
            length(W) < n && return (irreducible = false, dual_submodule = W)
            return (irreducible = true,)
        end
    end
    return (irreducible = missing,)
end

* The permutation module of $S_4$ on 4 points is reducible:

In [ ]:
pm = [F.(Matrix{Int}(I, 4, 4)[Perm(g).list, :]) for g in [[2,1,3,4], [1,3,2,4], [1,2,4,3]]]
norton(pm)

* The reflection representation of $S_4$, from its Cartan matrix (Part 2), is irreducible mod 7:

In [ ]:
C = [2 -1 0; -1 2 -1; 0 -1 2]
refl = [begin m = Matrix{Int}(I, 3, 3); m[:, s] = m[s, :] - C[s, :]; m end for s in 1:3]
norton([F.(m) for m in refl])

* The full MeatAxe also splits a reducible module into submodule and quotient, recursively, tests for **absolute** irreducibility, and computes module homomorphisms [HEO05].

## Exercises

**Exercise 5.1.**  Store the transversals of `Link` as **Schreier vectors**: for each point $y$ in the orbit, the generator $a$ with $y = z.a$ for the point $z$ from which $y$ was found.  Reconstruct $t_y$ when needed.  What is gained?

**Exercise 5.2.**  **Random Schreier-Sims.**  Instead of all Schreier generators, sift random elements of $G$, and stop when 20 of them in a row sift through.  Compare the running times on the Rubik's cube group.  Why can the result be wrong, and how can it be checked?

**Exercise 5.3.**  Use backtrack search to compute the **set stabilizer** of $\{1, 2, 3\}$ in $M_{24}$.  Which base makes the pruning effective?

**Exercise 5.4.**  Use `subgp_gens`, with the pruning of `centralizer`, to find generators of the centralizer of an element of order $23$ in $M_{24}$.  Add a count of the visited nodes, as in `backtrack`, and compare the two searches.

**Exercise 5.5.**  Implement the Tietze transformation that removes a generator $k$ occurring exactly once in some relator, and apply it to the presentations of $S_3 \leq S_4$ and of $\langle t_0, t_1, t_2 \rangle \leq G(3,3,3)$.

**Exercise 5.6.**  Use `norton` and `spin` to split the permutation module of $S_5$ on 5 points mod 7, and the Specht modules of Part 4.

## References

Besides those of the preface:

[HR94] D. F. Holt, S. Rees, Testing modules for irreducibility, *J. Austral. Math. Soc. Ser. A* **57** (1994), 1–16.

[Rei26] K. Reidemeister, Knoten und Gruppen, *Abh. Math. Sem. Univ. Hamburg* **5** (1926), 7–23.

[Sch27] O. Schreier, Die Untergruppen der freien Gruppen, *Abh. Math. Sem. Univ. Hamburg* **5** (1927), 161–183.